# S0 BP Online Runner

This notebook is a convenience wrapper around `scripts/run_baseline.py`.
It does not define a separate baseline path.

Intended use:
1. Open this notebook from the HPC Jupyter environment.
2. Set `EOH_LLM_API_KEY` in the configuration cell below.
3. Run all cells in order.

The notebook is currently set to use the prebaseline config by default for scale-up diagnosis.
Switch `ACTIVE_CONFIG_PATH` to the frozen baseline config only after the prebaseline path succeeds.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    candidates = [start, start.parent]
    for candidate in candidates:
        if (candidate / "scripts" / "run_baseline.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing scripts/run_baseline.py")


START_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(START_DIR)
BASELINE_CONFIG_PATH = REPO_ROOT / "configs" / "baseline_bp_online.yaml"
SMOKE_CONFIG_PATH = REPO_ROOT / "configs" / "smoke_bp_online.yaml"
PREBASELINE_CONFIG_PATH = REPO_ROOT / "configs" / "prebaseline_bp_online.yaml"
ACTIVE_CONFIG_PATH = PREBASELINE_CONFIG_PATH
RUNNER_PATH = REPO_ROOT / "scripts" / "run_baseline.py"
VERIFY_PATH = REPO_ROOT / "scripts" / "verify_run.py"
RUNS_DIR = REPO_ROOT / "runs"

print(f"Repo root: {REPO_ROOT}")
print(f"Baseline config: {BASELINE_CONFIG_PATH}")
print(f"Smoke config: {SMOKE_CONFIG_PATH}")
print(f"Prebaseline config: {PREBASELINE_CONFIG_PATH}")
print(f"Active config: {ACTIVE_CONFIG_PATH}")
print(f"Runner path: {RUNNER_PATH}")
print(f"Verify path: {VERIFY_PATH}")


In [ ]:
# Set the API key here for the current notebook session if it is not already in the environment.
# Do not save a real secret back into version control.

EOH_LLM_API_KEY = os.environ.get("EOH_LLM_API_KEY", "")

# If needed, paste the key into the next line before running the notebook on the HPC.
# EOH_LLM_API_KEY = "my-key-ensia-2022-1030"

if EOH_LLM_API_KEY:
    print("EOH_LLM_API_KEY is set for this notebook session.")
else:
    print("EOH_LLM_API_KEY is not set yet. Set it before running the baseline.")


In [ ]:
def run_python_script(script_path, *args):
    env = os.environ.copy()
    if EOH_LLM_API_KEY:
        env["EOH_LLM_API_KEY"] = EOH_LLM_API_KEY

    cmd = [sys.executable, str(script_path), *args]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=env,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with return code {completed.returncode}")
    return completed


def run_runner(*args):
    return run_python_script(RUNNER_PATH, "--config", str(ACTIVE_CONFIG_PATH), *args)


def validate_config():
    return run_runner("--validate-only")


def run_baseline(skip_posthoc=False):
    if not EOH_LLM_API_KEY:
        raise RuntimeError("EOH_LLM_API_KEY is empty. Set it in the notebook before running the baseline.")
    args = []
    if skip_posthoc:
        args.append("--skip-posthoc")
    return run_runner(*args)


def latest_run_dir():
    if not RUNS_DIR.exists():
        raise FileNotFoundError(f"Runs directory does not exist yet: {RUNS_DIR}")
    run_dirs = [path for path in RUNS_DIR.iterdir() if path.is_dir()]
    if not run_dirs:
        raise FileNotFoundError(f"No run directories found in {RUNS_DIR}")
    return max(run_dirs, key=lambda path: path.stat().st_mtime)


def inspect_latest_run():
    run_dir = latest_run_dir()
    print(f"Latest run: {run_dir}")

    expected = [
        run_dir / "run_manifest.json",
        run_dir / "run_summary.json",
        run_dir / "config_snapshot.yaml",
        run_dir / "logs" / "candidate_attempts.jsonl",
        run_dir / "logs" / "invalid_candidates.jsonl",
        run_dir / "logs" / "prompts",
        run_dir / "logs" / "responses",
        run_dir / "results" / "pops",
        run_dir / "results" / "pops_best",
        run_dir / "posthoc_eval" / "bp_online",
    ]

    checklist = []
    for path in expected:
        checklist.append({
            "path": str(path),
            "exists": path.exists(),
            "type": "dir" if path.exists() and path.is_dir() else "file",
        })

    print(json.dumps(checklist, indent=2))
    return run_dir, checklist


def verify_latest_run(expect_posthoc=True):
    run_dir = latest_run_dir()
    args = ["--run-dir", str(run_dir)]
    if expect_posthoc:
        args.append("--expect-posthoc")
    return run_python_script(VERIFY_PATH, *args)


## 1. Validate Config

This checks the active config, paths, endpoint, and model without launching a real run.

In [ ]:
validate_config()

## 2. Run Prebaseline Or Baseline

The notebook currently points to the prebaseline config by default.
Set `skip_posthoc=True` if you want a search-only prebaseline run.

In [ ]:
SKIP_POSTHOC = False
run_baseline(skip_posthoc=SKIP_POSTHOC)

## 3. Inspect Latest Run

This prints the latest run directory and a simple artifact checklist.

In [ ]:
inspect_latest_run()

## 4. Strict Verification

This validates the latest run folder against the expected S0 schema and cross-file consistency checks.

In [ ]:
verify_latest_run(expect_posthoc=not SKIP_POSTHOC)